In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from pathlib import Path
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

In [3]:
class FiLMMultimodalClassifier(nn.Module):
    def __init__(self, audio_dim=768, text_dim=312, hidden_dim=128, num_classes=3, dropout=0.5):
        super().__init__()

        # Аудио → параметры модуляции (scale и shift)
        self.audio_film = nn.Linear(audio_dim, hidden_dim * 2)

        # Текст → проекция в hidden_dim
        self.text_proj = nn.Linear(text_dim, hidden_dim)

        # Классификатор (очень лёгкий)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, audio_emb, text_emb):
        # audio_emb: [batch, 768]
        # text_emb: [batch, 312]

        # 1. Аудио генерирует scale и shift
        film_params = self.audio_film(audio_emb)  # [batch, 256]
        scale, shift = film_params.chunk(2, dim=1)  # [batch, 128], [batch, 128]

        # 2. Текст проецируем
        text = self.text_proj(text_emb)  # [batch, 128]

        # 3. Модуляция (внимание!)
        modulated = scale * text + shift  # [batch, 128]

        # 4. Классификация
        logits = self.classifier(modulated)

        return logits

In [4]:
def load_embeddings(file_path):
    """Загружает эмбеддинги из файла {имя: эмбеддинг}"""
    embeddings = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(maxsplit=1)
            if len(parts) == 2:
                name, emb_str = parts
                emb = np.array([float(x) for x in emb_str.split(',')])
                embeddings[name] = emb
    return embeddings

# Пути к файлам с эмбеддингами (твои)
audio_emb_path = "/content/drive/MyDrive/876_augmented/Emo_Emb.txt"
text_emb_path = "/content/drive/MyDrive/876_augmented/RuBert_Emb.txt"

print("Загрузка эмбеддингов...")
audio_embs = load_embeddings(audio_emb_path)
text_embs = load_embeddings(text_emb_path)
print(f"Аудио эмбеддингов: {len(audio_embs)}")
print(f"Текст эмбеддингов: {len(text_embs)}")

# Общие имена
common_names = set(audio_embs.keys()) & set(text_embs.keys())
print(f"Общих примеров: {len(common_names)}")

Загрузка эмбеддингов...
Аудио эмбеддингов: 3504
Текст эмбеддингов: 3504
Общих примеров: 3504


In [5]:
situation_list = ['threat', 'warning', 'neutral']
situation2id = {s: i for i, s in enumerate(situation_list)}
id2situation = {i: s for s, i in situation2id.items()}

X_audio = []
X_text = []
y = []

for name in common_names:
    parts = name.split('_')
    if len(parts) >= 3:
        situation = parts[2]  # 'threat', 'warning', 'neutral'
        if situation in situation2id:
            X_audio.append(audio_embs[name])
            X_text.append(text_embs[name])
            y.append(situation2id[situation])

print(f"Загружено {len(y)} примеров")
print(f"Распределение: {dict(zip(*np.unique(y, return_counts=True)))}")

Загружено 3316 примеров
Распределение: {np.int64(0): np.int64(820), np.int64(1): np.int64(832), np.int64(2): np.int64(1664)}


In [6]:
X_audio_train, X_audio_temp, X_text_train, X_text_temp, y_train, y_temp = train_test_split(
    X_audio, X_text, y, test_size=0.4, random_state=42, stratify=y
)

X_audio_val, X_audio_test, X_text_val, X_text_test, y_val, y_test = train_test_split(
    X_audio_temp, X_text_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}")

Train: 1989, Val: 663, Test: 664


In [7]:
class MultimodalDataset(Dataset):
    def __init__(self, audio_embs, text_embs, labels):
        self.audio_embs = [torch.tensor(emb, dtype=torch.float) for emb in audio_embs]
        self.text_embs = [torch.tensor(emb, dtype=torch.float) for emb in text_embs]
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.audio_embs[idx], self.text_embs[idx], self.labels[idx]

train_dataset = MultimodalDataset(X_audio_train, X_text_train, y_train)
val_dataset = MultimodalDataset(X_audio_val, X_text_val, y_val)
test_dataset = MultimodalDataset(X_audio_test, X_text_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FiLMMultimodalClassifier(num_classes=3)
model.to(device)

# Веса классов для борьбы с дисбалансом
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

print(f"Модель на {device}")
print(f"Параметров: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

best_val_loss = float('inf')
best_model_state = None

for epoch in range(100):
    # Train
    model.train()
    train_loss = 0
    for audio_emb, text_emb, labels in train_loader:
        audio_emb = audio_emb.to(device)
        text_emb = text_emb.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(audio_emb, text_emb)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    val_preds, val_true = [], []
    with torch.no_grad():
        for audio_emb, text_emb, labels in val_loader:
            audio_emb = audio_emb.to(device)
            text_emb = text_emb.to(device)
            labels = labels.to(device)

            logits = model(audio_emb, text_emb)
            loss = criterion(logits, labels)
            val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_true.extend(labels.cpu().numpy())

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    val_acc = accuracy_score(val_true, val_preds)

    scheduler.step(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = model.state_dict().copy()
        print(f"✅ Лучшая модель (Val Loss: {best_val_loss:.4f}, Val Acc: {val_acc:.4f})")

    print(f"Эпоха {epoch+1:2d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

Модель на cuda
Параметров: 245379
✅ Лучшая модель (Val Loss: 0.8658, Val Acc: 0.6848)
Эпоха  1 | Train Loss: 0.9791 | Val Loss: 0.8658 | Val Acc: 0.6848
✅ Лучшая модель (Val Loss: 0.7555, Val Acc: 0.7466)
Эпоха  2 | Train Loss: 0.8324 | Val Loss: 0.7555 | Val Acc: 0.7466
✅ Лучшая модель (Val Loss: 0.6906, Val Acc: 0.7617)
Эпоха  3 | Train Loss: 0.7548 | Val Loss: 0.6906 | Val Acc: 0.7617
✅ Лучшая модель (Val Loss: 0.6533, Val Acc: 0.7768)
Эпоха  4 | Train Loss: 0.7065 | Val Loss: 0.6533 | Val Acc: 0.7768
✅ Лучшая модель (Val Loss: 0.6266, Val Acc: 0.7662)
Эпоха  5 | Train Loss: 0.6691 | Val Loss: 0.6266 | Val Acc: 0.7662
✅ Лучшая модель (Val Loss: 0.6106, Val Acc: 0.7753)
Эпоха  6 | Train Loss: 0.6376 | Val Loss: 0.6106 | Val Acc: 0.7753
✅ Лучшая модель (Val Loss: 0.5949, Val Acc: 0.7828)
Эпоха  7 | Train Loss: 0.6115 | Val Loss: 0.5949 | Val Acc: 0.7828
✅ Лучшая модель (Val Loss: 0.5813, Val Acc: 0.8024)
Эпоха  8 | Train Loss: 0.5994 | Val Loss: 0.5813 | Val Acc: 0.8024
✅ Лучшая модел

In [9]:
model.load_state_dict(best_model_state)
torch.save(model, "/content/drive/MyDrive/film_multimodal_full.pth")
torch.save(best_model_state, "/content/drive/MyDrive/film_multimodal_weights.pth")
print("✅ Модель сохранена")

✅ Модель сохранена


In [10]:
model.eval()
test_preds, test_true = [], []
with torch.no_grad():
    for audio_emb, text_emb, labels in test_loader:
        audio_emb = audio_emb.to(device)
        text_emb = text_emb.to(device)
        labels = labels.to(device)

        logits = model(audio_emb, text_emb)
        preds = torch.argmax(logits, dim=1)
        test_preds.extend(preds.cpu().numpy())
        test_true.extend(labels.cpu().numpy())

test_acc = accuracy_score(test_true, test_preds)
print(f"\n{'='*50}")
print(f"Тест Accuracy: {test_acc:.4f}")
print(f"{'='*50}")
print("\nClassification Report:")
print(classification_report(test_true, test_preds, target_names=situation_list))


Тест Accuracy: 0.9292

Classification Report:
              precision    recall  f1-score   support

      threat       0.89      0.93      0.91       164
     warning       0.91      0.86      0.88       167
     neutral       0.96      0.96      0.96       333

    accuracy                           0.93       664
   macro avg       0.92      0.92      0.92       664
weighted avg       0.93      0.93      0.93       664

